# Diffusion Models Hands-On

17/04/2026

Deep lEarning a.y. 2025/2026


**Discaimer:**
What we will see today is a simplification with respect to the standard definition of Diffusion Models you saw in class. This model will learn how to reconstruct the original image from a corrupted version obtained after $t$ steps of noise injection. The model you will use today is a Unet ([Original Unet paper](https://arxiv.org/abs/1505.04597v1)), adapted for the task of image generation.

This was necessary as diffusion models are computationally very heavy, and require a lot of VRAM to work. We also implemented a simple _gradient accumulation_ scheme ([Gradient accumulation tutorial from Weights & Biases](https://wandb.ai/wandb_fc/tips/reports/How-To-Implement-Gradient-Accumulation-in-PyTorch--VmlldzoyMjMwOTk5)) to allow for larger batch-size during training.

Even with all these considerations, this implementation takes many hours to perform a full epoch of training on a Colab GPU runtime, therefore today we will guide you through the various steps and we'll then provide you an already trained model to be run in inference to see the generated outputs.

Do not forget to keep in mind the theory behind diffusion models while you complete this lab!

### A reminder of what diffusion models actually do

This is how the diffusion model works: it’s like based with a fully noisy image and gradually improves the image quality until it becomes clear. ( As below image showed )

![](https://miro.medium.com/v2/resize:fit:828/format:webp/1*CI0BnAddDdrljunrdFadyQ.png)

Therefore, we can create a Deep Learning model that can improve image quality ( from fully noise to clear image ), the flow idea:
![](https://miro.medium.com/v2/resize:fit:828/format:webp/1*m7twgbm24be19Md5AK6nfw.png)

The Training of this model is performed taking images and adding noise for t steps, and inputting the noisy image with the t inside the network. So each batch of training is composed of sampled images with different noise magnitude on it.

In this tutorial, we will be using a lot of flowers images from Flowers102.

### Directories set-up for downloading pre-trained model and storing models trained from scratch

In [ ]:
!pwd
import os
PRETRAINED_DIR = "./pretrained_models"
SCRATCH_DIR = "./fresh_models"

os.makedirs(PRETRAINED_DIR, exist_ok=True)
os.makedirs(SCRATCH_DIR, exist_ok=True)

# MODEL Download
!wget "https://www.dropbox.com/scl/fi/l2hrf7t42e108y3tagr9n/pretrained_models.zip?rlkey=bl85fxm896b0cqs9x2qclp3p5&dl=1"
!mv "pretrained_models.zip?rlkey=bl85fxm896b0cqs9x2qclp3p5&dl=1" pretrained_models.zip
!unzip pretrained_models.zip

In [ ]:
from tqdm.auto import trange, tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision import transforms

In [ ]:
IMG_SIZE = 256     # input image size
BATCH_SIZE = 8     # batch size for training
timesteps = 16      # how many diffusion steps
time_bar = 1 - torch.linspace(0, 1.0, timesteps + 1) # linspace for timesteps
BATCH_SIZE_INFERENCE= 4

device = torch.device('cuda')

### Here, we set “timesteps”, that is the number of steps for injection of noise and generation in our model

In [ ]:
plt.plot(time_bar, label='Noise')
plt.legend()
plt.show()

In [ ]:
# Prepare Flowers102

# code here for trasform, we want: 1. conversion to tensor, 2. Resizing to IMG_SIZE, 3. normalization with 0.5 0.5 0.5 mean and std

transform = # CODE HERE


all_trainset = torchvision.datasets.Flowers102(root='./data', split='train', download=True, transform=transform)
all_testset = torchvision.datasets.Flowers102(root='./data', split='test', download=True, transform=transform)

dataset = torch.utils.data.ConcatDataset([all_trainset, all_testset])


#Create a dataloader for the train_set, named trainloader, with batch_size = BATCH_SIZE.

trainloader = # CODE HERE

### Some images from dataset
Run the cell to see samples from dataset

In [ ]:
def show_samples(samples):
    plt.figure(figsize=(6, 3))
    for i, x in enumerate(samples):
        x = (x.permute(1, 2, 0) - torch.min(x))/ (torch.max(x) - torch.min(x))
        plt.subplot(2, 5, i+1)
        plt.grid(False)
        plt.axis("off")
        plt.imshow(x)
    plt.tight_layout()
    plt.show()

samples = torch.cat([all_trainset[i][0].unsqueeze(0) for i in torch.randint(0, len(all_trainset), size=(10, ))], dim=0)

show_samples(samples)

### Example of training samples: images at different trianing steps

In [ ]:
def forward_noise(x, t):
    a = time_bar[t]      # base on t
    b = time_bar[t + 1]  # image for t + 1

    noise =  # noise mask, use normal method from torch, assign mean and std correctly to get an image with full random noise
    a = a.reshape((-1, 1, 1, 1))
    b = b.reshape((-1, 1, 1, 1))
    img_a = x * (1 - a) + noise * a   #why is this? comment briefly
    img_b = x * (1 - b) + noise * b
    return img_a, img_b

def generate_ts(num):
    return  #code here to return a vector of random time steps (size "(num, )", values between 0 and timesteps)



Why we are generating img_a (with noise at time t) and a img_b (with noise at time t+1) ??

In [ ]:
print("Clean Images")
t = torch.full((10,), timesteps - 1) # if you want see original images
a, b = #use the function forward_noise giving samples and time steps
show_samples(a)

print("\nFull Noise")
t = torch.full((10,), 0)             # if you want see noisy
a, b =  #use the function forward_noise giving samples and time steps
show_samples(b)

print("\nIntermediate steps (x_t+1, x_t)")
t = # use the appropriate function to generate ten random time steps
a, b =  #use the function forward_noise giving samples and time steps
print("\nx_t+1")
show_samples(a)
print("\nx_t")
show_samples(b)


# Define model functions as a Unet
<figure>
<img src = "https://www.dropbox.com/scl/fi/w74dwwlbhm3futpomer84/Diff_Unet.png?rlkey=7yydpfxg9u1ozrodr56rj7xka&raw=1" >
<figcaption align="center"> Readapted from Uncertainty-based human-in-the-loop deep learning for land cover segmentation, by García Rodríguez et al. 2020.
</figure>


Each Block of the network takes as input the noisy image and the output is conditioned on the current time step.

Each block contains two convolutional layers with a time parameter, allowing the network to determine its current time step and output corresponding information.
You can see block flow chart:
( **x_img** is input image which is noisy image, **x_ts** is input for time step )

![](https://miro.medium.com/v2/resize:fit:828/format:webp/1*aBeNL-VIEYj4tgjkhMc0yw.png)



In [ ]:
class Block(nn.Module):
    def __init__(self, in_channels=128, size=IMG_SIZE):
        super(Block, self).__init__()

        self.conv_param = # define a Conv2d layer with in_channels=in_channels, 128 number of filters, kernel_size = 3 and padding = 1.
        self.conv_out = # define a Conv2d layer with in_channels=in_channels, 128 number of filters, kernel_size = 3 and padding = 1.

        self.dense_ts = # define a linear layer with 192 input neurons and 128 output ones for embedding the time steps


        self.layer_norm = nn.LayerNorm([128, size, size])

    def forward(self, x_img, x_ts):
        x_parameter = # apply relu to the input image given to conv_param layer

        time_parameter = # relu to the embedding of time step (so dense_ts layer)

        time_parameter = time_parameter.view(-1, 128, 1, 1)


        x_parameter = x_parameter * time_parameter

        x_out = #conv_out of input image
        x_out = x_out + x_parameter
        x_out = #relu applied to layer norm of x_out

        return x_out

### Build model.

![](https://miro.medium.com/v2/resize:fit:828/format:webp/1*jhqgZSzDeZx9YMR9QndmsQ.png)

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()

        self.l_ts = nn.Sequential(
            #linear layer that takes time step and embeds it (look at previous code for the embedding length)
            #layer norm
            #relu
        )

        self.down_x512 = Block(in_channels=3, size=IMG_SIZE)
        self.down_x256 = Block(size=IMG_SIZE//2)
        # complete the architecture up to self.down_x4, tip* you have to use block and modify size accordingly

        self.mlp = nn.Sequential(
            nn.Linear(2240, 128),
            nn.LayerNorm([128]),
            nn.ReLU(),

            nn.Linear(128, 32 * 4 * 4), # make [-1, 32, 4, 4]
            nn.LayerNorm([32 * 4 * 4]),
            nn.ReLU(),
        )

        self.up_x4 = Block(in_channels=32 + 128, size=IMG_SIZE//128)
        self.up_x8 = Block(in_channels=32 + 128, size=IMG_SIZE//64)
        self.up_x16 = Block(in_channels=256, size=IMG_SIZE//32)
        self.up_x32 = Block(in_channels=256, size=IMG_SIZE//16)
        self.up_x64 = Block(in_channels=256, size=IMG_SIZE//8)
        self.up_x128 = Block(in_channels=256, size=IMG_SIZE//4)
        self.up_x256 = Block(in_channels=256, size=IMG_SIZE//2)
        self.up_x512 = Block(in_channels=256, size=IMG_SIZE)

        self.cnn_output = nn.Conv2d(in_channels=128, out_channels=3, kernel_size=1, padding=0)

        # make optimizer, use Adam with a learning rate of 0.0008

    def forward(self, x, x_ts):
        x_ts = self.l_ts(x_ts)

        # ----- left ( down ) -----
        blocks = [
            self.down_x512,
            self.down_x256,
            self.down_x128,
            self.down_x64,
            self.down_x32,
            self.down_x16,
            self.down_x8,
            # self.down_x4,
        ]
        x_left_layers = []
        for i, block in enumerate(blocks):
            x = block(x, x_ts)
            x_left_layers.append(x)
            if i < len(blocks) - 1:
                x = F.max_pool2d(x, 2)

        # ----- MLP -----
        x = x.view(-1, 128 * 4 * 4)
        x = torch.cat([x, x_ts], dim=1)
        x = self.mlp(x)
        x = x.view(-1, 32, 4, 4)

        # ----- right ( up ) -----
        blocks = [
            # self.up_x4,
            self.up_x8,
            self.up_x16,
            self.up_x32,
            self.up_x64,
            self.up_x128,
            self.up_x256,
            self.up_x512,
        ]

        for i, block in enumerate(blocks):
            # cat left
            x_left = x_left_layers[len(blocks) - i - 1]
            x = torch.cat([x, x_left], dim=1)

            x = block(x, x_ts)
            if i < len(blocks) - 1:
                x = F.interpolate(x, scale_factor=2, mode='bilinear')

        # ----- output -----
        x = self.cnn_output(x)

        return x



## Generate sample images
We can now try our first generation. The steps for generation are as follows:



1.   create noisy images
2.   input to our model with time step
3.   keep doing this until end of time step





In [ ]:
model = Model().to(device)

In [ ]:
def predict(x_idx=None, batch_size=BATCH_SIZE):
 x = # generate a noisy image. tip* use randn with arguments batch_size and the shape of the image  (remember to copy x on the GPU)
 with torch.no_grad():
        for i in trange(timesteps):
            t = i
            x = model(x, torch.full([batch_size, 1], t, dtype=torch.float, device=device))

 show_samples(x.cpu())

predict(10)

Above is our Non-Trained Model output, as you can see, it’s nothing useful.

In [ ]:
def predict_step():
    xs = []
    x = # generate a noisy image. tip* use randn with arguments batch_size and the shape of the image

    with torch.no_grad():
        for i in trange(timesteps):
            t = i
            x = model(x, torch.full([BATCH_SIZE, 1], t, dtype=torch.float, device=device))
            if i % 2 == 0:
                xs.append(x[0].cpu())


    xs = torch.stack(xs, dim=0)
    xs = torch.clip(xs, -1, 1)
    show_samples(xs)



## Training Model

Now, we just need to train the model to learn how to reduce noise.
For that mission, we need two input in our model:

input image — the noise image need to be processed
timestamp — tell model what’s noise status so can be easier to learn


We simply need to provide **x_ts** and **x_img** to enable our model to learn how to generate denoised image.

In [ ]:
def train_one(x_img, minibatch_idx, num_minibatches):
    ACCUMULATION_STEPS = 4 # Number of accumulation steps
    x_ts = generate_ts(len(x_img))
    x_a, x_b = # generate x_a and x_b using the appropriate function

    x_ts = x_ts.view(-1, 1).float().to(device)
    x_a = x_a.float().to(device)
    x_b = x_b.float().to(device)

    y_p = #compute the output of our unet
    # Compute loss and scale it by the number of accumulation steps
    loss = # torch.mean applied to the difference of output image and ground_Truth. Divide by ACCUMULATION_STEPS for gradient cumulations.
    loss.backward()

    # Wait for the correct number of iterations before updating network parameters
    if ((minibatch_idx + 1) % ACCUMULATION_STEPS == 0) or (minibatch_idx + 1 == num_minibatches):
        model.opt.step()
        model.opt.zero_grad()

    return loss.item()

In [ ]:
#modify this (and the following block) just to train one epoch, but for the rest you can use the downlaoded model

def train(R=100):
    bar = trange(R)
    total = len(trainloader)
    for i in bar:
        model.opt.zero_grad()
        for j, (x_img, _) in enumerate(trainloader):
            loss = train_one(x_img, j, len(trainloader))
            pg = (j / total) * 100
            if j % 5 == 0:
                bar.set_description(f'loss: {loss:.5f}, p: {pg:.2f}%')

In [ ]:
for i in range(10):
    print(f'running {i}')
    train()
    # reduce learning rate for the next iteration
    for pg in model.opt.param_groups:
        pg['lr'] = max(0.000001, pg['lr'] * 0.9)

    # show result
    predict()
    predict_step()
    plt.show()
    torch.save(model.state_dict(), f'/fresh_models/{i}_flowers102.pt')

In [ ]:
torch.save(model.state_dict(), './fresh_models/final__flowers102.pt')
model.load_state_dict(torch.load('./fresh_models/final__flowers102.pt'))

# Load pre trained at different epochs

In [ ]:
for i in range(10):
    print(f'-----------------------------')
    print(f'runnig inference at epoch {i*100}')

    model.load_state_dict(torch.load(f'./{PRETRAINED_DIR}/{i}_flowers102.pt'))

    # show result
    #use the appropriate functions to generate some images, and then generate one image showing all the steps.
    plt.show()


In [ ]:
model.load_state_dict(torch.load(f'{PRETRAINED_DIR}/final__flowers102.pt'))

You can get some output images like this

In [ ]:
predict()
predict_step()

In [ ]:
def predict_similar(img, ts):

    xs = []

    x, _ = forward_noise(img, ts)

    x = x.float().to(device)

    with torch.no_grad():
        for i in range(ts, timesteps):
            t = i
            x = model(x, torch.full([BATCH_SIZE, 1], ts, dtype=torch.float, device=device))
            xs.append(x[0].cpu())
    xs = torch.stack(xs, dim=0)
    xs = torch.clip(xs, -1, 1)
    xs = (xs.permute(0, 2, 3, 1) - torch.min(xs)) / (torch.max(xs) - torch.min(xs))

    plt.figure(figsize=(20, 2))
    for i in range(len(xs)):
        plt.subplot(1, len(xs), i+1)
        plt.imshow(xs[i])
        plt.title(f'{i+ts}')
        plt.axis('off')

    plt.show()



t = 10


# use the iterator (next(iter(dataloader))) to get a batch of images

img, _ = next(iter(trainloader))

# use the function predict_similar giving a chosen image and the time step t to generate the image
predict_similar(img[0], t)

Try increasing the amount of noise (smaller time *t*)!

Same noisy image will end up in different denoised output.

In [ ]:
t = 5
img_index = 2000 # change image
img, _ = dataset[img_index]

predict_similar(img, t)
predict_similar(img, t)
predict_similar(img, t)
predict_similar(img, t)
predict_similar(img, t)